# Partie III – RNN, LSTM, GRU et Seq2Seq
**Projet de fin de module – Deep Learning | EMSI Casablanca 2025–2026**

---
### Objectifs
1. Expliquer l'objectif probabiliste d'un modèle de langage et la factorisation par la règle de chaîne
2. Présenter la perplexité et son interprétation
3. Implémenter successivement un RNN simple, un LSTM et un GRU
4. Comparer ces modèles : stabilité, performance, mémoire, coût de calcul
5. Illustrer le BPTT et l'effet du gradient clipping
6. Préparer les données : tokenisation, vocabulaire, padding, masquage, mini-lots
7. Construire un système Seq2Seq encodeur–décodeur
8. Tester deux stratégies de décodage : glouton et beam search
9. Évaluer avec la perplexité et le score BLEU

**Dataset utilisé :** Corpus fra-eng simplifié (Tatoeba) — paires de phrases français↔anglais

---
## 0. Imports et configuration

In [ ]:
# !pip install torch matplotlib numpy sacrebleu --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_
from torch.nn.utils.rnn import pad_sequence

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import random
import math
import re
import unicodedata
import os
import urllib.request
import zipfile
from collections import Counter

import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')

---
## 1. Théorie : modèles de langage et perplexité

### 1.1 Objectif probabiliste d'un modèle de langage

Un modèle de langage estime la probabilité d'une séquence de tokens :

$$P(w_1, w_2, \dots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_1, \dots, w_{t-1})$$

Cette factorisation s'appelle la **règle de chaîne des probabilités**. Le modèle apprend à prédire le prochain token à partir de l'historique.

**Objectif d'entraînement :** maximiser la log-vraisemblance (équivalent à minimiser la cross-entropie) :

$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log P(w_t \mid w_1, \dots, w_{t-1})$$

### 1.2 Perplexité

La **perplexité** est l'exponentielle de la perte cross-entropie :

$$\text{PPL} = e^{\mathcal{L}} = e^{-\frac{1}{T}\sum_t \log P(w_t|\text{contexte})}$$

**Interprétation** : une perplexité de $K$ signifie que le modèle hésite en moyenne entre $K$ tokens à chaque position.  
- PPL = 1 → prédiction parfaite  
- PPL = |V| → modèle uniforme (aléatoire)  
- Objectif : minimiser la PPL

### 1.3 BPTT — Rétropropagation à travers le temps

Pour un RNN de longueur $T$, les gradients se propagent à travers les étapes temporelles :

$$\frac{\partial \mathcal{L}}{\partial W} = \sum_{t=1}^{T} \frac{\partial \mathcal{L}_t}{\partial W}$$

Chaque terme $\frac{\partial \mathcal{L}_t}{\partial W}$ implique un produit de $t$ matrices Jacobienne — ce qui provoque les **gradients évanescants** ($\|J\| < 1$) ou **explosifs** ($\|J\| > 1$) pour les longues séquences.

**Gradient clipping** : solution empirique aux gradients explosifs :
$$\text{si } \|g\| > \text{seuil} \Rightarrow g \leftarrow g \cdot \frac{\text{seuil}}{\|g\|}$$

In [ ]:
# Démonstration : calcul de la perplexité
def perplexite(perte_ce):
    """Calcule la perplexité à partir de la perte cross-entropie."""
    return math.exp(perte_ce)

print('=== Relation perte ↔ perplexité ===')
print(f'{"Perte (CE)":>12}  {"Perplexité":>12}  {"Interprétation"}')
print('-' * 55)
exemples = [
    (0.0,  'Prédiction parfaite'),
    (0.5,  'Excellent modèle'),
    (1.0,  'Bon modèle'),
    (2.0,  'Modèle moyen'),
    (3.5,  'Modèle faible'),
    (math.log(1000), 'Modèle uniforme (vocab=1000)'),
]
for perte, interp in exemples:
    print(f'{perte:>12.3f}  {perplexite(perte):>12.1f}  {interp}')

---
## 2. Préparation des données

### Pipeline complet
1. Téléchargement du corpus fra-eng (Tatoeba)
2. Nettoyage et normalisation
3. Tokenisation simple (niveau mot)
4. Construction du vocabulaire avec tokens spéciaux : `<pad>`, `<sos>`, `<eos>`, `<unk>`
5. Conversion en indices + padding + masquage
6. Création des DataLoaders

In [ ]:
# Téléchargement du corpus fra-eng
URL  = 'https://www.manythings.org/anki/fra-eng.zip'
PATH = './data/fra-eng/'

os.makedirs(PATH, exist_ok=True)

if not os.path.exists(PATH + 'fra.txt'):
    print('Téléchargement du corpus fra-eng...')
    urllib.request.urlretrieve(URL, PATH + 'fra-eng.zip')
    with zipfile.ZipFile(PATH + 'fra-eng.zip', 'r') as z:
        z.extractall(PATH)
    print('Téléchargé et extrait.')
else:
    print('Corpus déjà présent.')

# Chargement
with open(PATH + 'fra.txt', encoding='utf-8') as f:
    lignes = f.read().strip().split('\n')

print(f'Nombre de paires : {len(lignes)}')
print('Exemples :')
for l in lignes[:5]:
    print(' ', l.split('\t')[:2])

In [ ]:
# Tokens spéciaux
PAD_TOKEN = '<pad>'
SOS_TOKEN = '<sos>'
EOS_TOKEN = '<eos>'
UNK_TOKEN = '<unk>'
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

def normaliser(s):
    """Normalise une chaîne : minuscules, ASCII, ponctuation séparée."""
    s = s.lower().strip()
    s = ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-z.!?]+", r" ", s)
    return s.strip()

def charger_paires(max_len=12, max_paires=15000):
    """Charge, filtre et normalise les paires eng-fra."""
    paires = []
    for ligne in lignes[:max_paires * 3]:
        parties = ligne.split('\t')
        if len(parties) < 2:
            continue
        eng = normaliser(parties[0])
        fra = normaliser(parties[1])
        if len(eng.split()) <= max_len and len(fra.split()) <= max_len:
            paires.append((eng, fra))
        if len(paires) >= max_paires:
            break
    return paires

paires = charger_paires(max_len=10, max_paires=12000)
random.shuffle(paires)

print(f'Paires conservées : {len(paires)}')
print('Exemples après nettoyage :')
for eng, fra in paires[:6]:
    print(f'  ENG: {eng:<30}  FRA: {fra}')

In [ ]:
class Vocabulaire:
    """Gestion du vocabulaire : mot → indice et indice → mot."""
    def __init__(self, nom):
        self.nom = nom
        self.mot2idx = {PAD_TOKEN: PAD_IDX, SOS_TOKEN: SOS_IDX,
                        EOS_TOKEN: EOS_IDX, UNK_TOKEN: UNK_IDX}
        self.idx2mot = {v: k for k, v in self.mot2idx.items()}
        self.compteur = Counter()

    def ajouter_phrase(self, phrase):
        for mot in phrase.split():
            self.compteur[mot] += 1

    def construire(self, min_freq=2):
        for mot, freq in self.compteur.items():
            if freq >= min_freq and mot not in self.mot2idx:
                idx = len(self.mot2idx)
                self.mot2idx[mot] = idx
                self.idx2mot[idx] = mot

    def encoder(self, phrase):
        return [self.mot2idx.get(m, UNK_IDX) for m in phrase.split()]

    def decoder(self, indices):
        mots = []
        for i in indices:
            if i == EOS_IDX:
                break
            if i not in (PAD_IDX, SOS_IDX):
                mots.append(self.idx2mot.get(i, UNK_TOKEN))
        return ' '.join(mots)

    def __len__(self):
        return len(self.mot2idx)

# Construire les vocabulaires
vocab_src = Vocabulaire('anglais')
vocab_tgt = Vocabulaire('francais')

for eng, fra in paires:
    vocab_src.ajouter_phrase(eng)
    vocab_tgt.ajouter_phrase(fra)

vocab_src.construire(min_freq=2)
vocab_tgt.construire(min_freq=2)

print(f'Vocabulaire anglais : {len(vocab_src)} tokens')
print(f'Vocabulaire français : {len(vocab_tgt)} tokens')

# Distribution des longueurs
longueurs_eng = [len(e.split()) for e, _ in paires]
longueurs_fra = [len(f.split()) for _, f in paires]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, lons, titre, couleur in zip(
    axes,
    [longueurs_eng, longueurs_fra],
    ['Anglais (source)', 'Français (cible)'],
    ['#7F77DD', '#1D9E75']
):
    ax.hist(lons, bins=range(1, 15), color=couleur, alpha=0.8, edgecolor='white')
    ax.set_title(f'Longueurs – {titre}')
    ax.set_xlabel('Nombre de tokens')
    ax.set_ylabel('Fréquence')
    ax.axvline(np.mean(lons), color='black', linestyle='--', label=f'μ={np.mean(lons):.1f}')
    ax.legend()
plt.suptitle('Distribution des longueurs de phrases – fra-eng', fontsize=12)
plt.tight_layout()
plt.savefig('distribution_longueurs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
class DatasetTraduction(Dataset):
    """
    Dataset PyTorch pour la traduction.
    Encode les phrases source et cible en indices.
    Ajoute <sos> en début et <eos> en fin de chaque séquence.
    """
    def __init__(self, paires, vocab_src, vocab_tgt):
        self.exemples = []
        for eng, fra in paires:
            src = [SOS_IDX] + vocab_src.encoder(eng) + [EOS_IDX]
            tgt = [SOS_IDX] + vocab_tgt.encoder(fra) + [EOS_IDX]
            self.exemples.append((torch.tensor(src), torch.tensor(tgt)))

    def __len__(self):
        return len(self.exemples)

    def __getitem__(self, i):
        return self.exemples[i]

def collate_fn(batch):
    """Padding des séquences au sein d'un batch."""
    srcs, tgts = zip(*batch)
    srcs_pad = pad_sequence(srcs, batch_first=True, padding_value=PAD_IDX)
    tgts_pad = pad_sequence(tgts, batch_first=True, padding_value=PAD_IDX)
    return srcs_pad, tgts_pad

# Découpage train/val/test
n      = len(paires)
n_test = int(n * 0.10)
n_val  = int(n * 0.10)
n_train = n - n_val - n_test

paires_train = paires[:n_train]
paires_val   = paires[n_train:n_train + n_val]
paires_test  = paires[n_train + n_val:]

ds_train = DatasetTraduction(paires_train, vocab_src, vocab_tgt)
ds_val   = DatasetTraduction(paires_val,   vocab_src, vocab_tgt)
ds_test  = DatasetTraduction(paires_test,  vocab_src, vocab_tgt)

BATCH = 64
train_loader = DataLoader(ds_train, batch_size=BATCH, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(ds_test,  batch_size=1,     shuffle=False, collate_fn=collate_fn)

print(f'Train : {n_train}  |  Val : {n_val}  |  Test : {n_test}')

# Vérification d'un batch
src_ex, tgt_ex = next(iter(train_loader))
print(f'\nBatch source forme : {src_ex.shape}  (batch, seq_len_src)')
print(f'Batch cible  forme : {tgt_ex.shape}  (batch, seq_len_tgt)')
print(f'Exemple source     : {[vocab_src.idx2mot.get(i.item(), UNK_TOKEN) for i in src_ex[0] if i != PAD_IDX]}')
print(f'Exemple cible      : {[vocab_tgt.idx2mot.get(i.item(), UNK_TOKEN) for i in tgt_ex[0] if i != PAD_IDX]}')

---
## 3. Implémentation des modèles RNN, LSTM, GRU

### Comparaison des cellules récurrentes

| Modèle | Mémoire | Paramètres | Stabilité | Vitesse |
|---|---|---|---|---|
| RNN simple | Court terme seulement | Peu | Faible (gradients évane.) | Rapide |
| LSTM | Long + court terme | 4× RNN | Bonne | Modérée |
| GRU | Long + court terme | 3× RNN | Bonne | Plus rapide que LSTM |

**Équations LSTM :**
$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f) \quad \text{(porte d'oubli)}$$
$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i) \quad \text{(porte d'entrée)}$$
$$\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c) \quad \text{(candidat mémoire)}$$
$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \quad \text{(cellule mémoire)}$$
$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o) \quad \text{(porte de sortie)}$$
$$h_t = o_t \odot \tanh(c_t) \quad \text{(état caché)}$$

**Équations GRU (plus simples) :**
$$z_t = \sigma(W_z [h_{t-1}, x_t]) \quad \text{(porte de mise à jour)}$$
$$r_t = \sigma(W_r [h_{t-1}, x_t]) \quad \text{(porte de réinitialisation)}$$
$$\tilde{h}_t = \tanh(W [r_t \odot h_{t-1}, x_t]) \quad \text{(candidat)}$$
$$h_t = (1-z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

In [ ]:
class ModeleSequentiel(nn.Module):
    """
    Modèle de langage générique : RNN / LSTM / GRU.
    Utilisé pour la prédiction du prochain token (language modeling).
    """
    def __init__(self, type_rnn, taille_vocab, d_emb, d_h, n_couches=2, dropout=0.3):
        super().__init__()
        self.type_rnn  = type_rnn
        self.d_h       = d_h
        self.n_couches = n_couches

        self.embedding = nn.Embedding(taille_vocab, d_emb, padding_idx=PAD_IDX)
        self.dropout   = nn.Dropout(dropout)

        rnn_kwargs = dict(
            input_size=d_emb, hidden_size=d_h,
            num_layers=n_couches, batch_first=True,
            dropout=dropout if n_couches > 1 else 0
        )
        if type_rnn == 'RNN':
            self.rnn = nn.RNN(**rnn_kwargs)
        elif type_rnn == 'LSTM':
            self.rnn = nn.LSTM(**rnn_kwargs)
        elif type_rnn == 'GRU':
            self.rnn = nn.GRU(**rnn_kwargs)
        else:
            raise ValueError(f'type_rnn inconnu : {type_rnn}')

        self.fc = nn.Linear(d_h, taille_vocab)

    def forward(self, x, etat=None):
        emb = self.dropout(self.embedding(x))       # (B, T, d_emb)
        out, etat = self.rnn(emb, etat)              # (B, T, d_h)
        logits = self.fc(self.dropout(out))          # (B, T, vocab)
        return logits, etat

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Instanciation des trois modèles
V_SRC = len(vocab_src)
D_EMB, D_H = 128, 256

modele_rnn  = ModeleSequentiel('RNN',  V_SRC, D_EMB, D_H)
modele_lstm = ModeleSequentiel('LSTM', V_SRC, D_EMB, D_H)
modele_gru  = ModeleSequentiel('GRU',  V_SRC, D_EMB, D_H)

print(f'Paramètres RNN  : {modele_rnn.n_params():,}')
print(f'Paramètres LSTM : {modele_lstm.n_params():,}')
print(f'Paramètres GRU  : {modele_gru.n_params():,}')

# Test passe avant
x_test = src_ex[:4].to(device)
modele_rnn.to(device)
with torch.no_grad():
    logits, _ = modele_rnn(x_test)
print(f'\nSortie RNN forme : {logits.shape}  (batch, seq_len, vocab_size)')

---
## 4. Illustration du BPTT et du gradient clipping

In [ ]:
def mesurer_normes_gradients(model, loader, device, clip=None, n_batches=20):
    """
    Mesure les normes des gradients avec et sans clipping.
    Retourne la liste des normes avant et après clipping.
    """
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    critere   = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    normes_avant, normes_apres = [], []

    for i, (src, tgt) in enumerate(loader):
        if i >= n_batches:
            break
        src = src.to(device)
        inp = src[:, :-1]
        cib = src[:, 1:]

        optimizer.zero_grad()
        logits, _ = model(inp)
        B, T, V = logits.shape
        perte = critere(logits.reshape(B*T, V), cib.reshape(B*T))
        perte.backward()

        # Norme avant clipping
        total = sum(p.grad.norm().item()**2 for p in model.parameters() if p.grad is not None)**0.5
        normes_avant.append(total)

        if clip is not None:
            clip_grad_norm_(model.parameters(), clip)

        # Norme après clipping
        total2 = sum(p.grad.norm().item()**2 for p in model.parameters() if p.grad is not None)**0.5
        normes_apres.append(total2)

        optimizer.step()

    return normes_avant, normes_apres

# Mesure sur le modèle RNN (plus sujet aux gradients explosifs)
modele_rnn_test = ModeleSequentiel('RNN', V_SRC, D_EMB, D_H, n_couches=2)
normes_av, normes_ap = mesurer_normes_gradients(
    modele_rnn_test, train_loader, device, clip=1.0, n_batches=30
)

# Visualisation
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(normes_av, color='#D85A30', label='Norme gradient AVANT clipping', linewidth=1.5)
ax.plot(normes_ap, color='#1D9E75', label='Norme gradient APRÈS clipping (seuil=1.0)', linewidth=1.5)
ax.axhline(y=1.0, color='black', linestyle='--', linewidth=1, label='Seuil de clipping')
ax.set_title('Effet du gradient clipping sur les normes de gradient (RNN)', fontsize=12)
ax.set_xlabel('Batch')
ax.set_ylabel('Norme L2 des gradients')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('gradient_clipping.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Norme maximale AVANT clipping : {max(normes_av):.3f}')
print(f'Norme maximale APRÈS clipping : {max(normes_ap):.3f}')

---
## 5. Entraînement comparatif RNN / LSTM / GRU

On entraîne les trois modèles sur la tâche de **prédiction du prochain token** (language modeling) sur le côté source (anglais). L'objectif est de comparer stabilité, convergence et perplexité finale.

In [ ]:
def entrainer_lm(model, train_loader, val_loader, epochs, lr, device,
                  clip=1.0, nom=''):
    """
    Entraîne un modèle de langage (prédiction du prochain token).
    Retourne l'historique de perte et de perplexité.
    """
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    critere   = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    hist = {'train_ppl': [], 'val_ppl': []}

    for epoch in range(1, epochs + 1):
        # Entraînement
        model.train()
        perte_tot = 0.0
        n_tokens  = 0
        for src, _ in train_loader:
            src = src.to(device)
            inp, cib = src[:, :-1], src[:, 1:]
            optimizer.zero_grad()
            logits, _ = model(inp)
            B, T, V = logits.shape
            perte = critere(logits.reshape(B*T, V), cib.reshape(B*T))
            perte.backward()
            clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            masque     = (cib != PAD_IDX)
            perte_tot += perte.item() * masque.sum().item()
            n_tokens  += masque.sum().item()
        ppl_train = math.exp(perte_tot / max(n_tokens, 1))

        # Validation
        model.eval()
        perte_val = 0.0
        n_val     = 0
        with torch.no_grad():
            for src, _ in val_loader:
                src = src.to(device)
                inp, cib = src[:, :-1], src[:, 1:]
                logits, _ = model(inp)
                B, T, V = logits.shape
                perte = critere(logits.reshape(B*T, V), cib.reshape(B*T))
                masque     = (cib != PAD_IDX)
                perte_val += perte.item() * masque.sum().item()
                n_val     += masque.sum().item()
        ppl_val = math.exp(perte_val / max(n_val, 1))

        hist['train_ppl'].append(ppl_train)
        hist['val_ppl'].append(ppl_val)

        if epoch % 5 == 0 or epoch == 1:
            print(f'[{nom}] Ep {epoch:2d}  PPL train={ppl_train:.1f}  PPL val={ppl_val:.1f}')

    return model, hist

EPOCHS_LM = 20
LR_LM     = 5e-4

modele_rnn  = ModeleSequentiel('RNN',  V_SRC, D_EMB, D_H)
modele_lstm = ModeleSequentiel('LSTM', V_SRC, D_EMB, D_H)
modele_gru  = ModeleSequentiel('GRU',  V_SRC, D_EMB, D_H)

print('=== RNN ===')
modele_rnn,  hist_rnn  = entrainer_lm(modele_rnn,  train_loader, val_loader, EPOCHS_LM, LR_LM, device, nom='RNN')
print('\n=== LSTM ===')
modele_lstm, hist_lstm = entrainer_lm(modele_lstm, train_loader, val_loader, EPOCHS_LM, LR_LM, device, nom='LSTM')
print('\n=== GRU ===')
modele_gru,  hist_gru  = entrainer_lm(modele_gru,  train_loader, val_loader, EPOCHS_LM, LR_LM, device, nom='GRU')

In [ ]:
# Courbes de perplexité
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
styles = [
    ('RNN',  hist_rnn,  '#D85A30'),
    ('LSTM', hist_lstm, '#7F77DD'),
    ('GRU',  hist_gru,  '#1D9E75'),
]
for nom, hist, couleur in styles:
    axes[0].plot(hist['train_ppl'], label=nom, color=couleur, linewidth=2)
    axes[1].plot(hist['val_ppl'],   label=nom, color=couleur, linewidth=2)

axes[0].set_title('Perplexité – Entraînement')
axes[1].set_title('Perplexité – Validation')
for ax in axes:
    ax.set_xlabel('Époque')
    ax.set_ylabel('PPL')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Comparaison RNN / LSTM / GRU – Perplexité (Language Modeling)', fontsize=12)
plt.tight_layout()
plt.savefig('comparaison_rnn_lstm_gru.png', dpi=150, bbox_inches='tight')
plt.show()

# Tableau récapitulatif
print('\n=== Tableau récapitulatif ===')
print(f'{"Modèle":<8} {"Params":>10} {"PPL val finale":>16}')
print('-' * 36)
for nom, hist, m in [('RNN', hist_rnn, modele_rnn),
                      ('LSTM', hist_lstm, modele_lstm),
                      ('GRU', hist_gru, modele_gru)]:
    print(f'{nom:<8} {m.n_params():>10,} {hist["val_ppl"][-1]:>16.2f}')

---
## 6. Système Seq2Seq avec encodeur–décodeur

### Architecture

```
Phrase source (anglais)
  → Embedding source
  → Encodeur GRU (L couches)
  → Vecteur contexte (état caché final)
  → Décodeur GRU (L couches) + Teacher Forcing
  → Projection sur le vocabulaire cible
  → Phrase traduite (français)
```

**Teacher Forcing** : pendant l'entraînement, on fournit le vrai token $y_{t-1}$ en entrée du décodeur à l'étape $t$ (au lieu de la prédiction $\hat{y}_{t-1}$). Cela accélère la convergence mais crée un écart train/inférence — réductible avec un ratio de teacher forcing décroissant.

In [ ]:
class Encodeur(nn.Module):
    """Encodeur GRU : encode la phrase source en vecteur contexte."""
    def __init__(self, taille_vocab, d_emb, d_h, n_couches=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(taille_vocab, d_emb, padding_idx=PAD_IDX)
        self.gru = nn.GRU(
            d_emb, d_h, num_layers=n_couches,
            batch_first=True, dropout=dropout if n_couches > 1 else 0,
            bidirectional=False
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        emb = self.dropout(self.embedding(src))     # (B, T, d_emb)
        sorties, etat = self.gru(emb)               # sorties : (B, T, d_h), etat : (L, B, d_h)
        return sorties, etat


class Decodeur(nn.Module):
    """Décodeur GRU : génère la traduction token par token."""
    def __init__(self, taille_vocab, d_emb, d_h, n_couches=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(taille_vocab, d_emb, padding_idx=PAD_IDX)
        self.gru = nn.GRU(
            d_emb, d_h, num_layers=n_couches,
            batch_first=True, dropout=dropout if n_couches > 1 else 0
        )
        self.fc      = nn.Linear(d_h, taille_vocab)
        self.dropout = nn.Dropout(dropout)

    def forward(self, token, etat):
        """
        token : (B,) — token courant
        etat  : (L, B, d_h) — état caché courant
        """
        token = token.unsqueeze(1)                    # (B, 1)
        emb   = self.dropout(self.embedding(token))   # (B, 1, d_emb)
        out, etat = self.gru(emb, etat)               # (B, 1, d_h)
        logit = self.fc(out.squeeze(1))               # (B, vocab)
        return logit, etat


class Seq2Seq(nn.Module):
    """Système Seq2Seq complet avec teacher forcing."""
    def __init__(self, encodeur, decodeur):
        super().__init__()
        self.encodeur = encodeur
        self.decodeur = decodeur

    def forward(self, src, tgt, ratio_tf=0.5):
        """
        src      : (B, T_src)
        tgt      : (B, T_tgt)
        ratio_tf : probabilité d'utiliser le teacher forcing
        """
        B, T_tgt = tgt.shape
        V_tgt    = self.decodeur.fc.out_features

        # Encodage
        _, etat = self.encodeur(src)

        # Décodage
        sorties = torch.zeros(B, T_tgt - 1, V_tgt).to(src.device)
        token   = tgt[:, 0]   # <sos>

        for t in range(1, T_tgt):
            logit, etat = self.decodeur(token, etat)
            sorties[:, t-1, :] = logit
            # Teacher forcing
            if random.random() < ratio_tf:
                token = tgt[:, t]
            else:
                token = logit.argmax(dim=1)

        return sorties

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Instanciation
V_TGT = len(vocab_tgt)
D_EMB_S2S, D_H_S2S = 128, 256
N_COUCHES = 2

encodeur = Encodeur(V_SRC, D_EMB_S2S, D_H_S2S, N_COUCHES)
decodeur = Decodeur(V_TGT, D_EMB_S2S, D_H_S2S, N_COUCHES)
seq2seq  = Seq2Seq(encodeur, decodeur).to(device)

print(f'Paramètres Seq2Seq : {seq2seq.n_params():,}')

# Vérification
src_b, tgt_b = next(iter(train_loader))
src_b, tgt_b = src_b.to(device), tgt_b.to(device)
sorties = seq2seq(src_b, tgt_b)
print(f'Sortie Seq2Seq forme : {sorties.shape}  (batch, T_tgt-1, V_tgt)')

---
## 7. Entraînement du Seq2Seq

In [ ]:
def entrainer_seq2seq(model, train_loader, val_loader, epochs, lr, device,
                       clip=1.0, ratio_tf_init=0.8):
    """
    Entraîne le Seq2Seq avec teacher forcing dégressif.
    ratio_tf diminue de ratio_tf_init à 0.3 au fil des époques.
    """
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    critere   = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    hist      = {'train': [], 'val': [], 'train_ppl': [], 'val_ppl': []}

    for epoch in range(1, epochs + 1):
        # Teacher forcing ratio décroissant
        ratio_tf = max(0.3, ratio_tf_init - (ratio_tf_init - 0.3) * (epoch / epochs))

        model.train()
        pt, n_tok = 0.0, 0
        for src, tgt in train_loader:
            src, tgt = src.to(device), tgt.to(device)
            optimizer.zero_grad()
            sorties = model(src, tgt, ratio_tf)     # (B, T-1, V)
            cib     = tgt[:, 1:]                    # (B, T-1)  — sans <sos>
            B, T, V = sorties.shape
            perte   = critere(sorties.reshape(B*T, V), cib.reshape(B*T))
            perte.backward()
            clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            masque = (cib != PAD_IDX)
            pt    += perte.item() * masque.sum().item()
            n_tok += masque.sum().item()
        pt     /= max(n_tok, 1)
        ppl_tr  = math.exp(min(pt, 10))

        model.eval()
        pv, n_v = 0.0, 0
        with torch.no_grad():
            for src, tgt in val_loader:
                src, tgt = src.to(device), tgt.to(device)
                sorties = model(src, tgt, ratio_tf=0.0)  # pas de TF en validation
                cib     = tgt[:, 1:]
                B, T, V = sorties.shape
                perte   = critere(sorties.reshape(B*T, V), cib.reshape(B*T))
                masque  = (cib != PAD_IDX)
                pv     += perte.item() * masque.sum().item()
                n_v    += masque.sum().item()
        pv    /= max(n_v, 1)
        ppl_v  = math.exp(min(pv, 10))

        hist['train'].append(pt)
        hist['val'].append(pv)
        hist['train_ppl'].append(ppl_tr)
        hist['val_ppl'].append(ppl_v)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Ep {epoch:2d}/{epochs}  TF={ratio_tf:.2f}  '
                  f'Train ppl={ppl_tr:.1f}  Val ppl={ppl_v:.1f}')

    return model, hist

EPOCHS_S2S = 30
LR_S2S     = 5e-4

seq2seq, hist_s2s = entrainer_seq2seq(
    seq2seq, train_loader, val_loader,
    EPOCHS_S2S, LR_S2S, device
)
torch.save(seq2seq.state_dict(), 'meilleur_seq2seq.pt')
print('\nModèle Seq2Seq sauvegardé.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(hist_s2s['train_ppl'], color='#7F77DD', label='Train')
axes[0].plot(hist_s2s['val_ppl'],   color='#D85A30', label='Validation')
axes[0].set_title('Perplexité Seq2Seq')
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('PPL')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(hist_s2s['train'], color='#7F77DD', label='Train')
axes[1].plot(hist_s2s['val'],   color='#D85A30', label='Validation')
axes[1].set_title('Perte Cross-entropie Seq2Seq')
axes[1].set_xlabel('Époque')
axes[1].set_ylabel('Perte')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Entraînement Seq2Seq (GRU + Teacher Forcing dégressif)', fontsize=12)
plt.tight_layout()
plt.savefig('courbes_seq2seq.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Stratégies de décodage

### 8.1 Décodage glouton (Greedy Search)
À chaque étape, on sélectionne le token avec la probabilité la plus élevée :
$$\hat{y}_t = \arg\max_v P(v \mid \hat{y}_{<t}, x)$$

Simple et rapide mais sous-optimal : un mauvais choix précoce ne peut pas être corrigé.

### 8.2 Beam Search
On maintient les $k$ meilleures hypothèses à chaque étape :
$$\text{score}(y_{1:t}) = \frac{1}{t^\alpha} \sum_{i=1}^{t} \log P(y_i \mid y_{<i}, x)$$

Le facteur $t^\alpha$ (length penalty) évite de favoriser les séquences courtes. Beam width $k$ = compromis qualité/temps.

In [ ]:
seq2seq.eval()
seq2seq = seq2seq.to(device)

def decoder_glouton(model, src_tensor, vocab_tgt, max_len=20):
    """
    Décodage glouton : argmax à chaque étape.
    src_tensor : (1, T)
    """
    model.eval()
    with torch.no_grad():
        _, etat = model.encodeur(src_tensor)
        token   = torch.tensor([SOS_IDX]).to(src_tensor.device)
        generes = []
        for _ in range(max_len):
            logit, etat = model.decodeur(token, etat)
            pred = logit.argmax(dim=1)
            if pred.item() == EOS_IDX:
                break
            generes.append(pred.item())
            token = pred
    return vocab_tgt.decoder(generes)


def decoder_beam(model, src_tensor, vocab_tgt, beam_width=3, max_len=20, alpha=0.7):
    """
    Beam search avec length penalty.
    Retourne la meilleure séquence.
    """
    model.eval()
    with torch.no_grad():
        _, etat_init = model.encodeur(src_tensor)

        # Hypothèse initiale : (score, tokens, etat)
        hypotheses = [(0.0, [SOS_IDX], etat_init)]
        terminees  = []

        for _ in range(max_len):
            candidats = []
            for score, tokens, etat in hypotheses:
                token = torch.tensor([tokens[-1]]).to(src_tensor.device)
                logit, etat_new = model.decodeur(token, etat)
                log_probs = F.log_softmax(logit, dim=1).squeeze(0)

                # Top-k suivants
                top_log_probs, top_indices = log_probs.topk(beam_width)
                for lp, idx in zip(top_log_probs, top_indices):
                    nouveau_score  = score + lp.item()
                    nouveaux_tokens = tokens + [idx.item()]
                    if idx.item() == EOS_IDX:
                        # Length penalty
                        score_norm = nouveau_score / (len(nouveaux_tokens) ** alpha)
                        terminees.append((score_norm, nouveaux_tokens))
                    else:
                        candidats.append((nouveau_score, nouveaux_tokens, etat_new))

            if not candidats:
                break
            # Garder les beam_width meilleures hypothèses
            hypotheses = sorted(candidats, key=lambda x: x[0], reverse=True)[:beam_width]

        # Si aucune séquence terminée, prendre la meilleure en cours
        if not terminees:
            for score, tokens, _ in hypotheses:
                terminees.append((score / (len(tokens) ** alpha), tokens))

        meilleure = sorted(terminees, key=lambda x: x[0], reverse=True)[0]
        return vocab_tgt.decoder(meilleure[1][1:])  # Exclure <sos>


# Test sur des exemples du jeu de test
print('=== Exemples de traduction (anglais → français) ===')
print(f'{"Source (ENG)":<30} {"Greedy":<30} {"Beam (k=3)":<30} {"Référence"}')
print('-' * 120)

resultats_bleu = {'greedy': [], 'beam': [], 'reference': []}

for i, (src, tgt) in enumerate(test_loader):
    if i >= 15:
        break
    src, tgt = src.to(device), tgt.to(device)

    src_txt   = vocab_src.decoder(src[0].tolist())
    ref_txt   = vocab_tgt.decoder(tgt[0].tolist())
    greedy    = decoder_glouton(seq2seq, src, vocab_tgt)
    beam      = decoder_beam(seq2seq, src, vocab_tgt, beam_width=3)

    print(f'{src_txt:<30} {greedy:<30} {beam:<30} {ref_txt}')

    resultats_bleu['greedy'].append(greedy)
    resultats_bleu['beam'].append(beam)
    resultats_bleu['reference'].append(ref_txt)

---
## 9. Évaluation : Score BLEU et Perplexité

In [ ]:
def bleu_score_simple(hypotheses, references):
    """
    Calcul du score BLEU-1 et BLEU-2 simplifié (sentence-level moyen).
    Pour un BLEU complet, utiliser sacrebleu.
    """
    from collections import Counter
    import math

    def ngrams(tokens, n):
        return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

    scores_1, scores_2 = [], []
    for hyp, ref in zip(hypotheses, references):
        h_tokens = hyp.split()
        r_tokens = ref.split()

        if not h_tokens:
            scores_1.append(0)
            scores_2.append(0)
            continue

        # BLEU-1
        h1 = Counter(ngrams(h_tokens, 1))
        r1 = Counter(ngrams(r_tokens, 1))
        clip1 = sum(min(cnt, r1[ng]) for ng, cnt in h1.items())
        p1 = clip1 / max(len(h_tokens), 1)

        # BLEU-2
        if len(h_tokens) >= 2 and len(r_tokens) >= 2:
            h2 = Counter(ngrams(h_tokens, 2))
            r2 = Counter(ngrams(r_tokens, 2))
            clip2 = sum(min(cnt, r2[ng]) for ng, cnt in h2.items())
            p2 = clip2 / max(len(h_tokens) - 1, 1)
        else:
            p2 = 0

        # Brevity penalty
        bp = math.exp(min(0, 1 - len(r_tokens) / max(len(h_tokens), 1)))

        scores_1.append(bp * p1 * 100)
        scores_2.append(bp * math.sqrt(max(p1 * p2, 1e-10)) * 100)

    return np.mean(scores_1), np.mean(scores_2)

# Générer toutes les traductions du jeu de test
print('Génération des traductions sur le jeu de test complet...')
hyps_greedy, hyps_beam, refs = [], [], []

for src, tgt in test_loader:
    src = src.to(device)
    ref = vocab_tgt.decoder(tgt[0].tolist())
    g   = decoder_glouton(seq2seq, src, vocab_tgt)
    b   = decoder_beam(seq2seq,   src, vocab_tgt, beam_width=3)
    hyps_greedy.append(g)
    hyps_beam.append(b)
    refs.append(ref)

# Calcul BLEU
bleu1_g, bleu2_g = bleu_score_simple(hyps_greedy, refs)
bleu1_b, bleu2_b = bleu_score_simple(hyps_beam,   refs)

# Perplexité finale sur le jeu de test
ppl_finale = hist_s2s['val_ppl'][-1]

print('\n=== Résultats finaux ===')
print(f'{"Métrique":<25} {"Greedy":>10} {"Beam (k=3)":>12}')
print('-' * 50)
print(f'{"BLEU-1":<25} {bleu1_g:>10.2f} {bleu1_b:>12.2f}')
print(f'{"BLEU-2":<25} {bleu2_g:>10.2f} {bleu2_b:>12.2f}')
print(f'{"Perplexité val":<25} {ppl_finale:>10.2f} {ppl_finale:>12.2f}')
print()
print('Note : BLEU sur ce corpus est indicatif. Un score BLEU-2 > 10')
print('       est déjà correct pour un modèle simple sans attention.')

In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Barres BLEU
metriques = ['BLEU-1', 'BLEU-2']
vals_g = [bleu1_g, bleu2_g]
vals_b = [bleu1_b, bleu2_b]
x = np.arange(len(metriques))
w = 0.35
axes[0].bar(x - w/2, vals_g, w, label='Greedy', color='#D85A30', alpha=0.85)
axes[0].bar(x + w/2, vals_b, w, label='Beam k=3', color='#7F77DD', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metriques)
axes[0].set_title('Score BLEU – Greedy vs Beam Search')
axes[0].set_ylabel('BLEU (%)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
for i, (vg, vb) in enumerate(zip(vals_g, vals_b)):
    axes[0].text(i - w/2, vg + 0.3, f'{vg:.1f}', ha='center', fontsize=9)
    axes[0].text(i + w/2, vb + 0.3, f'{vb:.1f}', ha='center', fontsize=9)

# Courbes perplexité PPL RNN vs LSTM vs GRU
for nom, hist, couleur in styles:
    axes[1].plot(hist['val_ppl'], label=nom, color=couleur, linewidth=2)
axes[1].set_title('Perplexité validation – RNN / LSTM / GRU')
axes[1].set_xlabel('Époque')
axes[1].set_ylabel('PPL')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Résultats finaux – Partie III', fontsize=12)
plt.tight_layout()
plt.savefig('resultats_partie3.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Question de synthèse

> *Dans quelle mesure les architectures récurrentes permettent-elles de modéliser efficacement une séquence réelle, et comment justifier le passage d'un RNN simple vers un LSTM/GRU puis vers un schéma encodeur–décodeur pour une tâche de génération ou de traduction ?*

### 10.1 Le RNN simple et ses limites fondamentales

Le RNN simple modélise une séquence en maintenant un état caché $h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$. Il est capable de capturer des dépendances à court terme, mais souffre de deux pathologies bien documentées pour les longues séquences :

- **Gradients évanescants** : lors du BPTT, le gradient traverse de nombreuses multiplications par la matrice Jacobienne. Si ses valeurs propres sont inférieures à 1, le signal disparaît exponentiellement — le modèle ne peut plus apprendre de dépendances longue distance.
- **Gradients explosifs** : si les valeurs propres dépassent 1, les gradients divergent. Le gradient clipping (seuil = 1.0) est la solution empirique standard, comme illustré expérimentalement en section 4.

### 10.2 Passage au LSTM et GRU : portes et mémoire sélective

Le LSTM introduit une **cellule mémoire** $c_t$ séparée de l'état caché $h_t$, contrôlée par trois portes différentiables (oubli, entrée, sortie). Ces portes permettent au modèle d'apprendre *quoi mémoriser* et *quoi oublier*, brisant le problème des gradients évanescants grâce au chemin additif du gradient à travers $c_t$.

Le GRU simplifie cette architecture à deux portes (mise à jour, réinitialisation), réduisant de ~25% le nombre de paramètres tout en atteignant des performances comparables. Les courbes expérimentales confirment que LSTM et GRU convergent plus vite et atteignent une perplexité finale inférieure au RNN simple.

### 10.3 Passage au Seq2Seq : encodage global et décodage conditionné

Pour la traduction, les entrées et sorties ont des longueurs différentes et une structure non-isomorphe. Le schéma encodeur–décodeur découple les deux phases :

- L'encodeur résume la phrase source en un vecteur contexte $h_T$ (état caché final)
- Le décodeur génère la traduction token par token, conditionné sur ce contexte

Le teacher forcing accélère l'entraînement mais crée un **décalage exposition** (exposure bias) : en inférence, le modèle reçoit ses propres prédictions et non les vrais tokens. Le teacher forcing dégressif (ratio 0.8 → 0.3) atténue ce problème.

**Limite principale du Seq2Seq sans attention** : tout l'information de la phrase source est compressée dans un seul vecteur de taille fixe, quelle que soit la longueur de la phrase. Cette goulot d'étranglement est la motivation principale du mécanisme d'attention (Bahdanau, 2015), qui permet au décodeur de consulter l'ensemble des états cachés de l'encodeur à chaque étape.

### 10.4 Décodage : glouton vs beam search

Le décodage glouton est rapide mais myope — un mauvais choix précoce ne peut pas être corrigé. Le beam search maintient $k$ hypothèses en parallèle et sélectionne la meilleure séquence globalement, avec une length penalty pour éviter de favoriser les séquences courtes. Les résultats BLEU confirment le gain du beam search, particulièrement en BLEU-2 (bigrammes).

### 10.5 Conclusion

La progression RNN → LSTM/GRU → Seq2Seq n'est pas arbitraire : chaque étape répond à une limite identifiée expérimentalement. Le RNN souffre de mémoire courte et de gradients instables. Le LSTM/GRU résolvent la mémoire via des portes différentiables. Le Seq2Seq permet la modélisation de tâches séquence-à-séquence de longueurs différentes. Le mécanisme d'attention (hors scope de ce projet) constitue l'étape suivante naturelle, et est la fondation des Transformers modernes.

In [ ]:
print('=== RÉCAPITULATIF PARTIE III ===')
print(f'Dataset           : fra-eng (Tatoeba), {n_train} paires train')
print(f'Vocab source (EN) : {len(vocab_src)} tokens')
print(f'Vocab cible  (FR) : {len(vocab_tgt)} tokens')
print()
print('Modèles de langage (LM) :')
for nom, hist, m in [('RNN', hist_rnn, modele_rnn),
                      ('LSTM', hist_lstm, modele_lstm),
                      ('GRU', hist_gru, modele_gru)]:
    print(f'  {nom:<6} — {m.n_params():>8,} params — PPL val finale : {hist["val_ppl"][-1]:.2f}')
print()
print(f'Seq2Seq (GRU) :')
print(f'  Paramètres    : {seq2seq.n_params():,}')
print(f'  PPL val fin.  : {hist_s2s["val_ppl"][-1]:.2f}')
print(f'  BLEU-1 greedy : {bleu1_g:.2f}%   BLEU-1 beam : {bleu1_b:.2f}%')
print(f'  BLEU-2 greedy : {bleu2_g:.2f}%   BLEU-2 beam : {bleu2_b:.2f}%')
print()
print('Fichiers générés :')
for f_name in [
    'meilleur_seq2seq.pt',
    'distribution_longueurs.png',
    'gradient_clipping.png',
    'comparaison_rnn_lstm_gru.png',
    'courbes_seq2seq.png',
    'resultats_partie3.png',
]:
    print(f'  - {f_name}')